# Fine-tuning a masked language model

## Load Dataset:

In [1]:
from datasets import load_dataset

raw_datasets = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

## Load Tokenizer:

In [2]:
from transformers import AutoTokenizer

ckpt = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(ckpt)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

## Load Model:

In [3]:
from transformers import AutoModelForMaskedLM
import torch

ckpt = "distilbert-base-uncased"

model = AutoModelForMaskedLM.from_pretrained(
    ckpt, 
    attn_implementation="flash_attention_3", 
    dtype=torch.bfloat16,
    device_map="auto"
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] You are attempting to use Flash Attention 3 with dropout. This might lead to unexpected behaviour as this is not supported on recent versions of Flash Attention.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## Tokenize Dataset:

In [4]:
tokenized_datasets = raw_datasets.map(
    function=lambda x: tokenizer(x['text'], truncation=True, max_length=512), 
    batched=True, 
    remove_columns=["text", "label"]
)
tokenized_datasets

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})

## Data Collator:

In [5]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

## Domain Adapt Model:

In [6]:
from transformers import TrainingArguments
from transformers import Trainer

In [7]:
args = TrainingArguments(
    output_dir="distilbert-mlm-imdb",
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
    per_device_train_batch_size=256,
    bf16=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["unsupervised"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [8]:
import math

perplexity = math.exp(trainer.evaluate()["eval_loss"])
perplexity

Training Loss,Validation Loss,Epoch
No log,2.837168,0


17.067353751596368

In [9]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,2.327325
2,No log,2.322924
3,2.415480,2.315938


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=588, training_loss=2.412182788459622, metrics={'train_runtime': 322.8793, 'train_samples_per_second': 464.57, 'train_steps_per_second': 1.821, 'total_flos': 1.98841734144e+16, 'train_loss': 2.412182788459622, 'epoch': 3.0})

In [10]:
import math

perplexity = math.exp(trainer.evaluate()["eval_loss"])
perplexity

Training Loss,Validation Loss,Epoch
2.415480,2.320278,3


10.178502817216474

In [12]:
from huggingface_hub import create_repo

# 1. Define your model ID
repo_id = "tankgauravgt/distilbert-cased-imdb-finetuned"
trainer.hub_model_id = repo_id

# 2. Create the repository on the Hub (safe if it already exists)
create_repo(repo_id=repo_id, exist_ok=True)

# 3. Push your model and logs to the Hub
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/tankgauravgt/distilbert-cased-imdb-finetuned/commit/14d8883ae055ca8bf64f57410a31e434d710c94f', commit_message='End of training', commit_description='', oid='14d8883ae055ca8bf64f57410a31e434d710c94f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tankgauravgt/distilbert-cased-imdb-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='tankgauravgt/distilbert-cased-imdb-finetuned'), pr_revision=None, pr_num=None)